In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import create_engine
import pymysql


def extract_marketcap_from_dataguide(
    excel_path,
    start_date=None,
    end_date=None,
    sheet_name=0
):
    """
    DataGuide 엑셀 파일에서 시가총액 데이터를 long format으로 변환

    Parameters:
    -----------
    excel_path : str
        DataGuide 엑셀 파일 경로
    start_date : str, optional
        추출 시작 날짜 (형식: 'YYYY-MM-DD')
    end_date : str, optional
        추출 종료 날짜 (형식: 'YYYY-MM-DD')
    sheet_name : int or str, default=0
        시트 이름 또는 인덱스

    Returns:
    --------
    df_long : DataFrame
        long format 데이터프레임 (date, ticker, indicator, value)
    """

    # 1. 엑셀 파일 읽기 (헤더 없이)
    df_raw = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)

    print("원본 데이터 shape:", df_raw.shape)

    # 2. 날짜 데이터 추출 (A14 = row 13, column 0부터 시작)
    dates = df_raw.iloc[13:, 0].copy()
    dates = pd.to_datetime(dates, errors='coerce')
    dates = dates.dropna()

    # 3. Ticker 추출 (A9 = row 8, column 1부터 가로로)
    tickers = df_raw.iloc[8, 1:].copy()
    tickers = tickers.dropna()

    print(f"\n추출된 날짜 개수: {len(dates)}")
    print(f"날짜 범위: {dates.min()} ~ {dates.max()}")
    print(f"\n추출된 Ticker 개수: {len(tickers)}")
    print(f"Ticker 샘플: {tickers[:5].tolist()}")

    # 4. 시가총액 데이터 추출
    marketcap_data = df_raw.iloc[13:13+len(dates), 1:1+len(tickers)].copy()

    # 5. 데이터프레임 구성
    marketcap_data.index = dates
    marketcap_data.columns = tickers

    # 6. 날짜 필터링
    if start_date:
        start_date = pd.to_datetime(start_date)
        marketcap_data = marketcap_data[marketcap_data.index >= start_date]
        print(f"\n시작 날짜 필터: {start_date}")

    if end_date:
        end_date = pd.to_datetime(end_date)
        marketcap_data = marketcap_data[marketcap_data.index <= end_date]
        print(f"종료 날짜 필터: {end_date}")

    print(f"\n필터링 후 날짜 개수: {len(marketcap_data)}")

    if len(marketcap_data) == 0:
        print("⚠️ 필터링 후 데이터가 없습니다.")
        return pd.DataFrame(columns=['date', 'ticker', 'indicator', 'value'])

    print(f"필터링 후 날짜 범위: {marketcap_data.index.min()} ~ {marketcap_data.index.max()}")

    # 7. Long format으로 변환 (수정된 부분)
    df_long = marketcap_data.reset_index()
    df_long = df_long.melt(
        id_vars=df_long.columns[0],  # 첫 번째 컬럼 (날짜)
        value_vars=df_long.columns[1:],  # 나머지 컬럼들 (ticker들)
        var_name='ticker',
        value_name='value'
    )

    # 8. 컬럼명 정리
    df_long.columns = ['date', 'ticker', 'value']

    # 9. indicator 컬럼 추가
    df_long['indicator'] = '시가총액'

    # 10. 컬럼 순서 조정
    df_long = df_long[['date', 'ticker', 'indicator', 'value']]

    # 11. NaN 제거
    original_len = len(df_long)
    df_long = df_long.dropna(subset=['value'])
    print(f"\nNaN 제거: {original_len - len(df_long):,}개 행 제거")

    # 12. 중복 제거
    original_len = len(df_long)
    df_long = df_long.drop_duplicates(subset=['date', 'ticker'], keep='first')
    duplicates_removed = original_len - len(df_long)

    if duplicates_removed > 0:
        print(f"⚠️ 중복 데이터 제거: {duplicates_removed:,}개 행")
    else:
        print("✓ 중복 데이터 없음")

    # 13. 정수형 변환
    def safe_int_convert(val):
        if pd.isna(val):
            return val
        try:
            if np.isfinite(val) and val == int(val):
                return int(val)
            return val
        except (ValueError, TypeError, OverflowError):
            return val

    df_long['value'] = df_long['value'].apply(safe_int_convert)

    # 14. 정렬
    df_long = df_long.sort_values(['date', 'ticker']).reset_index(drop=True)

    # 15. 최종 결과 출력
    print("\n" + "="*60)
    print("변환 완료!")
    print("="*60)
    print(f"총 레코드 수: {len(df_long):,}")
    print(f"고유 날짜 수: {df_long['date'].nunique():,}")
    print(f"고유 Ticker 수: {df_long['ticker'].nunique():,}")

    if len(df_long) > 0:
        print(f"날짜 범위: {df_long['date'].min()} ~ {df_long['date'].max()}")

        # 중복 검증
        dup_check = df_long.groupby(['date', 'ticker']).size()
        if (dup_check > 1).any():
            print("\n⚠️ 경고: 중복된 date-ticker 조합이 있습니다!")
            print(dup_check[dup_check > 1])
        else:
            print("\n✓ 중복 검증 완료: 모든 date-ticker 조합이 유일합니다")

    return df_long

def save_marketcap_to_db(
    excel_path,
    db_info,
    table_name='ks_listed_company_daily_marketcap',
    start_date=None,
    end_date=None,
    if_exists='append'
):
    """
    DataGuide 시가총액 데이터를 DB에 저장

    Parameters:
    -----------
    excel_path : str
        DataGuide 엑셀 파일 경로
    db_info : dict
        데이터베이스 연결 정보
        {
            'host': 'localhost',
            'port': 3306,
            'user': 'your_username',
            'password': 'your_password',
            'database': 'investar',
            'charset': 'utf8mb4'
        }
    table_name : str
        테이블 이름 (스키마 제외)
    start_date : str, optional
        추출 시작 날짜
    end_date : str, optional
        추출 종료 날짜
    if_exists : str, default='append'
        'append' 또는 'replace'

    Returns:
    --------
    df_long : DataFrame
        저장된 데이터프레임
    """

    # 1. 데이터 추출
    print("="*60)
    print("1단계: 엑셀 데이터 추출")
    print("="*60)

    df_long = extract_marketcap_from_dataguide(
        excel_path=excel_path,
        start_date=start_date,
        end_date=end_date
    )

    if len(df_long) == 0:
        print("\n추출된 데이터가 없습니다.")
        return df_long

    # 2. DB 연결
    print("\n" + "="*60)
    print("2단계: 데이터베이스 연결")
    print("="*60)

    try:
        # SQLAlchemy 엔진 생성
        connection_string = (
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
            f"@{db_info['host']}:{db_info.get('port', 3306)}"
            f"/{db_info['database']}?charset={db_info.get('charset', 'utf8mb4')}"
        )

        engine = create_engine(connection_string)

        print(f"✓ 데이터베이스 연결 성공")
        print(f"  - Host: {db_info['host']}")
        print(f"  - Database: {db_info['database']}")
        print(f"  - Table: {table_name}")

    except Exception as e:
        print(f"✗ 데이터베이스 연결 실패: {e}")
        return None

    # 3. 중복 체크 및 저장
    print("\n" + "="*60)
    print("3단계: 데이터베이스 저장")
    print("="*60)

    try:
        if if_exists == 'append':
            print("\n중복 데이터 체크 중...")

            # 기존 데이터 확인
            check_query = f"""
            SELECT date, ticker
            FROM {table_name}
            WHERE indicator = '시가총액'
            AND date BETWEEN '{df_long['date'].min()}' AND '{df_long['date'].max()}'
            """

            try:
                existing_data = pd.read_sql(check_query, engine)

                if len(existing_data) > 0:
                    existing_keys = set(
                        zip(
                            pd.to_datetime(existing_data['date']).dt.date,
                            existing_data['ticker']
                        )
                    )
                    new_keys = set(
                        zip(
                            pd.to_datetime(df_long['date']).dt.date,
                            df_long['ticker']
                        )
                    )

                    duplicates = existing_keys & new_keys

                    if duplicates:
                        print(f"⚠️ {len(duplicates):,}개 중복 발견")
                        print("중복 데이터를 제외하고 저장합니다...")

                        # 중복 제거
                        df_long['key'] = list(
                            zip(
                                pd.to_datetime(df_long['date']).dt.date,
                                df_long['ticker']
                            )
                        )
                        df_long = df_long[~df_long['key'].isin(duplicates)]
                        df_long = df_long.drop('key', axis=1)

                        print(f"저장할 데이터: {len(df_long):,} rows")
                    else:
                        print("✓ 중복 없음")
                else:
                    print("✓ 기존 데이터 없음 (전체 저장)")

            except Exception as e:
                print(f"기존 데이터 확인 중 오류: {e}")
                print("모든 데이터를 저장합니다...")

        # DB에 저장
        if len(df_long) > 0:
            df_long.to_sql(
                name=table_name,
                con=engine,
                if_exists=if_exists,
                index=False,
                method='multi',
                chunksize=1000
            )

            print(f"\n✓ DB 저장 완료!")
            print(f"  - 저장된 레코드: {len(df_long):,} rows")
            print(f"  - 테이블: {db_info['database']}.{table_name}")

            # 저장 후 검증
            verify_query = f"""
            SELECT COUNT(*) as cnt
            FROM {table_name}
            WHERE indicator = '시가총액'
            AND date BETWEEN '{df_long['date'].min()}' AND '{df_long['date'].max()}'
            """
            result = pd.read_sql(verify_query, engine)
            print(f"  - 검증: 해당 기간 총 {result['cnt'].iloc[0]:,}개 레코드")

        else:
            print("\n저장할 새 데이터가 없습니다.")

        engine.dispose()

    except Exception as e:
        print(f"\n✗ 저장 실패: {e}")
        engine.dispose()
        raise

    return df_long

In [5]:
from DATA.stock_invest_function import fetch_table_data, get_db_host

# 사용 예시 1: 데이터 추출만 (이 부분은 선택사항)
# df_marketcap = extract_marketcap_from_dataguide(
#     excel_path=r'C:\Users\82108\OneDrive\바탕 화면\investment\data\raw_data\월간시가총액.xlsx',
#     start_date='2025-06-01',
#     end_date='2025-12-31'
# )

# 사용 예시 2: DB에 바로 저장
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar',
    'charset': 'utf8mb4'
}

# DB에 저장 (내부에서 extract_marketcap_from_dataguide를 호출함)
df_saved = save_marketcap_to_db(
    excel_path=r'C:\Users\82108\OneDrive\바탕 화면\investment\data\raw_data\월간시가총액_20263011.xlsx',
    db_info=db_info,
    table_name='ks_listed_company_daily_marketcap',
    start_date='2025-06-01',
    end_date='2026-03-31',
    if_exists='append'  # 'append': 기존 데이터에 추가, 'replace': 전체 교체
)

# 결과 확인
if df_saved is not None and len(df_saved) > 0:
    print("\n저장된 데이터 샘플:")
    print(df_saved.head(20))

    # 통계 확인
    print(f"\n저장된 데이터 통계:")
    print(f"- 총 레코드: {len(df_saved):,}개")
    print(f"- 날짜 범위: {df_saved['date'].min()} ~ {df_saved['date'].max()}")
    print(f"- Ticker 수: {df_saved['ticker'].nunique()}개")

    # 특정 종목 확인 (삼성전자)
    if 'A005930' in df_saved['ticker'].values:
        print("\n삼성전자(A005930) 데이터:")
        print(df_saved[df_saved['ticker'] == 'A005930'].head())
else:
    print("\n저장된 데이터가 없거나 저장에 실패했습니다.")

1단계: 엑셀 데이터 추출
원본 데이터 shape: (126, 2507)

추출된 날짜 개수: 112
날짜 범위: 2016-12-29 00:00:00 ~ 2026-03-11 00:00:00

추출된 Ticker 개수: 2506
Ticker 샘플: ['A005930', 'A000660', 'A005380', 'A373220', 'A207940']

시작 날짜 필터: 2025-06-01 00:00:00
종료 날짜 필터: 2026-03-31 00:00:00

필터링 후 날짜 개수: 10
필터링 후 날짜 범위: 2025-06-30 00:00:00 ~ 2026-03-11 00:00:00

NaN 제거: 267개 행 제거
✓ 중복 데이터 없음

변환 완료!
총 레코드 수: 24,793
고유 날짜 수: 10
고유 Ticker 수: 2,504
날짜 범위: 2025-06-30 00:00:00 ~ 2026-03-11 00:00:00

✓ 중복 검증 완료: 모든 date-ticker 조합이 유일합니다

2단계: 데이터베이스 연결
✓ 데이터베이스 연결 성공
  - Host: 192.168.0.230
  - Database: investar
  - Table: ks_listed_company_daily_marketcap

3단계: 데이터베이스 저장

중복 데이터 체크 중...
⚠️ 17,284개 중복 발견
중복 데이터를 제외하고 저장합니다...
저장할 데이터: 7,509 rows

✓ DB 저장 완료!
  - 저장된 레코드: 7,509 rows
  - 테이블: investar.ks_listed_company_daily_marketcap
  - 검증: 해당 기간 총 7,509개 레코드

저장된 데이터 샘플:
            date   ticker indicator        value
17284 2026-01-30  A000020      시가총액    174571.69
17285 2026-01-30  A000040      시가총액     38955.21
17286 2026